In [1]:
!pip -q install datasets pandas

In [2]:
from datasets import load_dataset

base = "https://huggingface.co/datasets/PolyAI/banking77/resolve/main/data"
banking = load_dataset(
    "parquet",
    data_files={
        "train":    f"{base}/train-00000-of-00001.parquet", 
        "test":     f"{base}/test-00000-of-00001.parquet",
        },
)


print(banking)
print(banking['train'][0])
print(banking['test'][0])

labels_names = banking['train'].features['label'].names
print("no_of_intents:", len(labels_names))
print("first_examples_intent:", labels_names[banking["train"][0]["label"]])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}
{'text': 'How do I locate my card?', 'label': 11}
no_of_intents: 77
first_examples_intent: card_arrival


In [ ]:
from collections import Counter

label_names = banking["train"].features["label"].names

for split in ["train", "test"]:
    counts = Counter(banking[split]["label"])
    empty_messages = sum(
        not text.strip() for text in banking[split]["text"]
    )

    print(f"\n{split.upper()}")
    print("Messages:", len(banking[split]))
    print("Intents represented:", len(counts))
    print("Fewest examples in an intent:", min(counts.values()))
    print("Most examples in an intent:", max(counts.values()))
    print("Empty messages:", empty_messages)

print("\nFive example messages:")
for index in [0, 100, 500, 1000, 2000]:
    row = banking["train"][index]
    print(f"{index}: [{label_names[row['label']]}] {row['text']}")


TRAIN
Messages: 10003
Intents represented: 77
Fewest examples in an intent: 35
Most examples in an intent: 187
Empty messages: 0

TEST
Messages: 3080
Intents represented: 77
Fewest examples in an intent: 40
Most examples in an intent: 40
Empty messages: 0

Five example messages:
0: [card_arrival] I am still waiting on my card?
100: [card_arrival] can you share card tracking number?
500: [card_payment_wrong_exchange_rate] Why is the exchange rate different from what I expected?
1000: [fiat_currency_support] What are the currency types that I can keep money in.
2000: [pending_top_up] My top is still pending.


In [ ]:
from collections import defaultdict
import random

SEED = 42

def balanced_indices(split, examples_per_intent, seed):
    by_intent = defaultdict(list)

    for index, label in enumerate(split["label"]):
        by_intent[label].append(index)

    rng = random.Random(seed)
    selected = []
    for label in range(len(by_intent)):
        selected.extend(
        rng.sample(by_intent[label], examples_per_intent)
        )
    return sorted(selected)

dev_indices = balanced_indices(banking["train"], 1, SEED)
final_indices = balanced_indices(banking["train"], 2, SEED)


dev_set = banking["train"].select(dev_indices)
final_set = banking["train"].select(final_indices)

assert len(dev_set) == 77
assert len(final_set) == 154
assert len(dev_set["label"]) == 77
assert len(final_set["label"]) == 154

print("seed:", SEED)
print("dev set length:", len(dev_set))
print("final set length:", len(final_set))
print(" first 5 dev indices:", dev_indices[:5])
print(" first 5 final indices:", final_indices[:5])


seed: 42
dev set length: 77
final set length: 154
 first 5 dev indices: [108, 160, 381, 533, 658]
 first 5 final indices: [107, 139, 154, 224, 321]


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))